# Terra Signal: discretização pelo método AID e criação das dummies (Grupo 1)

Estudo de como transformar as variáveis preditoras em dummies. As numéricas são discretizadas com o método AID, usando o critério do qui-quadrado.

Cada pergunta tem três células: **pergunta** → **código** → **resposta**.

- **Numéricas estudadas:** `tenure` e `TotalCharges`. `MonthlyIncome` e `MonthlyCharges` não entram como preditoras, por decisão do grupo.
- **Limpeza:** regras do notebook 06.
- **`CustomerFeedback`:** fica de fora, por vazar o alvo (pergunta 30 do notebook 03).
- Roda com pandas, numpy, scipy e scikit-learn, sem Spark. Funciona localmente e no Databricks (Git folder).

## Setup

In [1]:
import warnings

import numpy as np
import pandas as pd
from scipy.stats import chi2, chi2_contingency
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

DATA_DIR = "../data/raw/"  # este notebook fica em notebooks/; os CSVs ficam em data/raw/
hist_raw = pd.read_csv(DATA_DIR + "history.csv")
inf_raw = pd.read_csv(DATA_DIR + "inference.csv")
print("history:", hist_raw.shape, "| inference:", inf_raw.shape)

history: (5634, 23) | inference: (1409, 22)


In [2]:
def prep(d):
    """Regras do notebook 06: padroniza PhoneService, recupera tenure e imputa TotalCharges."""
    d = d.copy()
    d["PhoneService"] = d["PhoneService"].str.capitalize()
    d["tenure_num"] = pd.to_numeric(d["tenure"], errors="coerce")
    d["tenure_num"] = d["tenure_num"].fillna((d["TotalCharges"] / d["MonthlyCharges"]).round())
    d["TotalCharges_imp"] = d["TotalCharges"].fillna(d["MonthlyCharges"] * d["tenure_num"])
    d.loc[d["tenure_num"] == 0, "TotalCharges_imp"] = 0
    return d


hist = prep(hist_raw)
hist = hist[hist["tenure_num"].notna()].reset_index(drop=True)  # 1 linha sem tenure e sem TotalCharges
inf = prep(inf_raw)
y = (hist["Churn"] == "Yes").astype(int)
print("Clientes no treino:", len(hist), "| churners:", y.sum(), f"({y.mean():.1%})")

Clientes no treino: 5633 | churners: 1503 (26.7%)


## Como o AID é aplicado

O AID procura os cortes que **mais separam** churners de não churners dentro de cada faixa, segundo o qui-quadrado:

1. Para cada corte possível, monta uma tabela 2×2 (abaixo ou acima do corte × churn) e calcula o qui-quadrado.
2. Divide a faixa no corte de maior qui-quadrado, começando pela faixa em que a separação é maior.
3. Repete em cada nova faixa até uma destas condições: o corte deixa de ser significativo (p < 0,05, com correção de Bonferroni pelo número de cortes testados), a faixa ficaria com menos de 5% da base, ou atingiu o limite de faixas.

O limite de 5% e o nível de significância são escolhas deste estudo, e a pergunta 2 testa o número de faixas.

**Sobre o nome:** o AID original (Morgan e Sonquist, 1963) usa redução de variância, e o uso do qui-quadrado é mais associado ao CHAID (Kass, 1980). Seguimos o critério do livro. A implementação abaixo faz divisões binárias sucessivas.

In [3]:
def melhor_corte(x, y, min_folha):
    """Corte binário de maior qui-quadrado, com p-valor corrigido por Bonferroni pelo nº de cortes testados."""
    tab = pd.DataFrame({"x": x, "y": y}).groupby("x")["y"].agg(["size", "sum"]).sort_index()
    n_esq, c_esq = tab["size"].cumsum().values, tab["sum"].cumsum().values
    n, c = n_esq[-1], c_esq[-1]
    validos = (n_esq >= min_folha) & (n - n_esq >= min_folha)
    if not validos.any():
        return None
    a, b = c_esq, n_esq - c_esq
    e, f = c - c_esq, (n - n_esq) - (c - c_esq)
    den = (a + b) * (e + f) * (a + e) * (b + f)
    with np.errstate(divide="ignore", invalid="ignore"):
        q = np.where(den > 0, n * (a * f - b * e) ** 2 / den, 0.0)
    q = np.where(validos, q, -1.0)
    i = int(np.argmax(q))
    return float(tab.index[i]), float(q[i]), min(1.0, chi2.sf(q[i], 1) * int(validos.sum()))


def aid(x, y, min_folha=0.05, alfa=0.05, max_faixas=None):
    """Divide sempre a faixa com maior qui-quadrado, até faltar significância, tamanho ou o limite de faixas."""
    x, y = np.asarray(x, float), np.asarray(y)
    tam = int(np.ceil(min_folha * len(x)))
    nos, passos = [np.ones(len(x), bool)], []
    while not max_faixas or len(nos) < max_faixas:
        candidatos = []
        for k, m in enumerate(nos):
            r = melhor_corte(x[m], y[m], tam)
            if r and r[2] < alfa:
                candidatos.append((r[1], k, r))
        if not candidatos:
            break
        _, k, r = max(candidatos, key=lambda z: z[0])
        m = nos.pop(k)
        nos += [m & (x <= r[0]), m & (x > r[0])]
        passos.append({"corte": r[0], "qui2": round(r[1], 1), "p_bonferroni": r[2]})
    return sorted(p["corte"] for p in passos), passos


def faixas(x, cortes):
    return pd.cut(x, [-np.inf] + list(cortes) + [np.inf])



DUMMIES_CATEG = {
    "gender": ["Female"],
    "SeniorCitizen": [1],
    "Partner": ["Yes"],
    "Dependents": ["Yes"],
    "MultipleLines": ["Yes", "No phone service"],
    "InternetService": ["DSL", "No"],
    "OnlineSecurity": ["Yes"],
    "OnlineBackup": ["Yes"],
    "DeviceProtection": ["Yes"],
    "TechSupport": ["Yes"],
    "StreamingTV": ["Yes"],
    "StreamingMovies": ["Yes"],
    "Contract": ["One year", "Two year"],
    "PaperlessBilling": ["No"],
    "PaymentMethod": ["Mailed check", "Credit card (automatic)", "Bank transfer (automatic)"],
}


def dummies_categoricas(d):
    """Uma coluna 0/1 para cada categoria listada. A categoria omitida é a referência."""
    return pd.DataFrame(
        {f"{col}={val}": (d[col] == val).astype(int) for col, vals in DUMMIES_CATEG.items() for val in vals},
        index=d.index,
    )



def dummies_faixas(x, cortes, nome, ref):
    """Uma dummy por faixa, exceto a faixa de referência `ref` (posição da faixa)."""
    f = np.digitize(x, cortes, right=True)
    limites = [-np.inf] + list(cortes) + [np.inf]
    return pd.DataFrame(
        {f"{nome}=({limites[i]:g}, {limites[i + 1]:g}]": (f == i).astype(int) for i in range(len(limites) - 1) if i != ref},
        index=x.index,
    )


def faixa_referencia(x, cortes):
    """A referência é a faixa mais frequente."""
    return int(pd.Series(np.digitize(x, cortes, right=True)).value_counts().idxmax())

In [4]:
def iv(f, y):
    g = pd.DataFrame({"f": pd.Series(f).astype(str).values, "y": np.asarray(y)}).groupby("f")["y"].agg(["size", "sum"])
    pb, pm = (g["size"] - g["sum"]) / (g["size"] - g["sum"]).sum(), g["sum"] / g["sum"].sum()
    return float(((pb - pm) * np.log(pb / pm)).sum())


def tabela(x, y, cortes):
    f = faixas(x, cortes)
    t = pd.DataFrame({"clientes": y.groupby(f).size(), "churners": y.groupby(f).sum(), "taxa_churn": y.groupby(f).mean().round(3)})
    t["pct_base"] = (t["clientes"] / t["clientes"].sum()).round(3)
    return t

### 1. Que cortes o AID encontra em `tenure`, sem limite de faixas?

In [5]:
cortes_livre, passos = aid(hist["tenure_num"], y)
print(pd.DataFrame(passos).to_string(index=False))
print()
print(tabela(hist["tenure_num"], y, cortes_livre).to_string())
print("\nIV:", round(iv(faixas(hist["tenure_num"], cortes_livre), y), 4))

 corte  qui2  p_bonferroni
  17.0 552.0 3.235002e-120
  43.0 117.4  1.164603e-25
   5.0  83.9  6.203993e-19
  59.0  33.0  2.044134e-07
  70.0  17.2  2.346440e-04
   1.0  13.4  4.993236e-04



              clientes  churners  taxa_churn  pct_base
tenure_num                                            
(-inf, 1.0]        486       293       0.603     0.086
(1.0, 5.0]         606       298       0.492     0.108
(5.0, 17.0]       1023       351       0.343     0.182
(17.0, 43.0]      1485       353       0.238     0.264
(43.0, 59.0]       853       126       0.148     0.151
(59.0, 70.0]       774        71       0.092     0.137
(70.0, inf]        406        11       0.027     0.072

IV: 0.8173


**Resposta:** sem limite, o AID gera 7 faixas, com cortes em 1, 5, 17, 43, 59 e 70 meses. O corte mais forte é o de 17 meses (qui-quadrado de 552). O churn cai de forma contínua, de 60,3% até 1 mês para 2,7% acima de 70 meses, e o IV é de 0,82.

### 2. Quantas faixas de `tenure` usar?

Comparamos o desempenho **fora da amostra**. O AID usa o alvo, então os cortes são recalculados no treino de cada fold para não inflar o resultado. O modelo é uma regressão logística com todas as dummies categóricas e as de `tenure` (5 folds, 4 repetições).

In [6]:
X_cat = dummies_categoricas(hist)
CV = RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=0)


def bloco(v, nome, tr, te, cortes):
    ref = faixa_referencia(hist[v].iloc[tr], cortes)
    return dummies_faixas(hist[v].iloc[tr], cortes, nome, ref), dummies_faixas(hist[v].iloc[te], cortes, nome, ref)


def avaliar(config):
    """config: lista de (variável, nome, função que devolve os cortes dado o índice de treino)."""
    aucs = []
    for tr, te in CV.split(hist, y):
        pa, pb = [X_cat.iloc[tr]], [X_cat.iloc[te]]
        for v, nome, f_cortes in config:
            a, b = bloco(v, nome, tr, te, f_cortes(tr))
            pa.append(a); pb.append(b)
        m = LogisticRegression(max_iter=3000).fit(pd.concat(pa, axis=1), y.iloc[tr])
        aucs.append(roc_auc_score(y.iloc[te], m.predict_proba(pd.concat(pb, axis=1))[:, 1]))
    return np.array(aucs)


def por_aid(v, max_faixas=None):
    return lambda tr: aid(hist[v].iloc[tr], y.iloc[tr], max_faixas=max_faixas)[0]


def por_decis(v):
    return lambda tr: list(np.unique(np.quantile(hist[v].iloc[tr], np.arange(.1, 1, .1))))


ref_livre = avaliar([("tenure_num", "tenure", por_aid("tenure_num"))])
linhas = []
for k in [2, 3, 4, 5, 6]:
    a = avaliar([("tenure_num", "tenure", por_aid("tenure_num", k))])
    linhas.append({"faixas": k, "cortes (base toda)": aid(hist["tenure_num"], y, max_faixas=k)[0], "AUC": round(a.mean(), 4),
                   "dif_vs_sem_limite": round(np.mean(a - ref_livre), 4), "dp_dif": round(np.std(a - ref_livre), 4)})
linhas.append({"faixas": "sem limite", "cortes (base toda)": cortes_livre, "AUC": round(ref_livre.mean(), 4), "dif_vs_sem_limite": 0.0, "dp_dif": 0.0})
print(pd.DataFrame(linhas).to_string(index=False))
sem_tempo = avaliar([])
print(f"\nSem nenhuma variável de tempo: AUC {sem_tempo.mean():.4f} | diferença {np.mean(sem_tempo - ref_livre):+.4f}")

    faixas                 cortes (base toda)    AUC  dif_vs_sem_limite  dp_dif
         2                             [17.0] 0.8359            -0.0097  0.0036
         3                       [17.0, 43.0] 0.8369            -0.0086  0.0032
         4                  [5.0, 17.0, 43.0] 0.8428            -0.0027  0.0018
         5            [5.0, 17.0, 43.0, 59.0] 0.8433            -0.0023  0.0015
         6      [5.0, 17.0, 43.0, 59.0, 70.0] 0.8448            -0.0007  0.0014
sem limite [1.0, 5.0, 17.0, 43.0, 59.0, 70.0] 0.8455             0.0000  0.0000



Sem nenhuma variável de tempo: AUC 0.8255 | diferença -0.0201


**Resposta:** **6 faixas**, com cortes em 5, 17, 43, 59 e 70 meses. O critério foi o menor número de faixas cuja diferença para a versão sem limite fica dentro de um desvio-padrão: −0,0007 ± 0,0014, com uma dummy a menos.

- **Com 4 faixas**, como no exemplo do livro, a perda é pequena, mas passa desse critério: −0,0027 ± 0,0018. Quatro faixas não são uma regra do método, só o resultado do exemplo.
- **Sem nenhuma variável de tempo**, a AUC cai 0,020. Esse é o limite do que está em jogo se o `tenure` estiver contaminado pelo momento do cancelamento, dúvida ainda não respondida pela TS.

A célula abaixo mostra as faixas finais.

In [7]:
FAIXAS_TENURE = 6
CORTES_TENURE = aid(hist["tenure_num"], y, max_faixas=FAIXAS_TENURE)[0]
print("Cortes finais de tenure:", CORTES_TENURE)
print(tabela(hist["tenure_num"], y, CORTES_TENURE).to_string())
print("\nIV com as faixas do AID:", round(iv(faixas(hist["tenure_num"], CORTES_TENURE), y), 4))
print("IV com decis:", round(iv(pd.qcut(hist["tenure_num"], 10, duplicates="drop"), y), 4))

Cortes finais de tenure: [5.0, 17.0, 43.0, 59.0, 70.0]
              clientes  churners  taxa_churn  pct_base
tenure_num                                            
(-inf, 5.0]       1092       591       0.541     0.194
(5.0, 17.0]       1023       351       0.343     0.182
(17.0, 43.0]      1485       353       0.238     0.264
(43.0, 59.0]       853       126       0.148     0.151
(59.0, 70.0]       774        71       0.092     0.137
(70.0, inf]        406        11       0.027     0.072

IV com as faixas do AID: 0.8045
IV com decis: 0.7682


### 3. Os cortes de `tenure` são estáveis?

In [8]:
rng = np.random.RandomState(0)
amostras = []
for _ in range(100):
    i = rng.randint(0, len(hist), len(hist))
    amostras.append(aid(hist["tenure_num"].iloc[i], y.iloc[i], max_faixas=FAIXAS_TENURE)[0])
print("100 reamostragens com reposição | parcela com um corte a até 2 meses do corte final:")
for c in CORTES_TENURE:
    print(f"  corte {c:4.0f}: {np.mean([any(abs(x - c) <= 2 for x in a) for a in amostras]):.0%}")

print("\nA proporção em cada faixa é parecida no inference?")
print(pd.DataFrame({
    "history": faixas(hist["tenure_num"], CORTES_TENURE).value_counts(normalize=True, sort=False).round(3),
    "inference": faixas(inf["tenure_num"], CORTES_TENURE).value_counts(normalize=True, sort=False).round(3),
}).to_string())

t = pd.to_numeric(hist_raw["tenure"], errors="coerce")
ok = t.notna() & hist_raw["TotalCharges"].notna() & (t > 0)
est = (hist_raw["TotalCharges"] / hist_raw["MonthlyCharges"]).round()
print("\nO tenure recuperado cai na mesma faixa do real?",
      f"{(np.digitize(est[ok], CORTES_TENURE, right=True) == np.digitize(t[ok], CORTES_TENURE, right=True)).mean():.1%}")

100 reamostragens com reposição | parcela com um corte a até 2 meses do corte final:
  corte    5: 81%
  corte   17: 57%
  corte   43: 58%
  corte   59: 51%
  corte   70: 79%

A proporção em cada faixa é parecida no inference?
              history  inference
tenure_num                      
(-inf, 5.0]     0.194      0.199
(5.0, 17.0]     0.182      0.172
(17.0, 43.0]    0.264      0.258
(43.0, 59.0]    0.151      0.156
(59.0, 70.0]    0.137      0.125
(70.0, inf]     0.072      0.090

O tenure recuperado cai na mesma faixa do real? 94.5%


**Resposta:** os cortes são razoavelmente estáveis, e os dos extremos são os mais firmes. Em 100 reamostragens, um corte a até 2 meses de 5 e de 70 aparece em cerca de 80% das vezes. Para 17, 43 e 59, fica entre 51% e 58%, pois esses cortes se deslocam mais.

Essa oscilação pesa pouco no resultado, já que a pergunta 2 mostra que a AUC quase não muda com as faixas. As faixas também se aplicam bem às outras situações:

- **Inference:** a maior diferença na proporção de clientes por faixa é de 1,8 ponto percentual.
- **`tenure` recuperado (notebook 06):** cai na mesma faixa do valor real em 94,5% dos casos.

### 4. Que cortes o AID encontra em `TotalCharges`?

In [9]:
cortes_tc, passos_tc = aid(hist["TotalCharges_imp"], y)
print(pd.DataFrame(passos_tc).to_string(index=False))
print()
t = tabela(hist["TotalCharges_imp"], y, cortes_tc)
f = faixas(hist["TotalCharges_imp"], cortes_tc)
t["tenure_medio"] = hist["tenure_num"].groupby(f).mean().round(1)
t["mensalidade_media"] = hist["MonthlyCharges"].groupby(f).mean().round(1)
print(t.to_string())
print("\nIV:", round(iv(f, y), 4))

rng = np.random.RandomState(0)
amostras = [aid(hist["TotalCharges_imp"].iloc[i], y.iloc[i])[0] for i in (rng.randint(0, len(hist), len(hist)) for _ in range(100))]
print("\nParcela de reamostragens com um corte a até 10% do corte final:")
for c in cortes_tc:
    print(f"  corte {c:8.2f}: {np.mean([any(abs(x - c) <= .1 * c for x in a) for a in amostras]):.0%}")

  corte  qui2  p_bonferroni
 387.20 247.2  5.152209e-52
3762.00  57.9  1.016038e-10
  96.05  43.4  3.057075e-08
  58.30  35.5  1.501256e-07

                  clientes  churners  taxa_churn  pct_base  tenure_medio  mensalidade_media
TotalCharges_imp                                                                           
(-inf, 58.3]           349       145       0.415     0.062           1.2               30.6
(58.3, 96.05]          295       192       0.651     0.052           1.8               59.6
(96.05, 387.2]         737       256       0.347     0.131           6.6               47.9
(387.2, 3762.0]       2847       705       0.248     0.505          31.7               59.4
(3762.0, inf]         1405       205       0.146     0.249          61.2               93.5



IV: 0.3415



Parcela de reamostragens com um corte a até 10% do corte final:
  corte    58.30: 44%
  corte    96.05: 82%
  corte   387.20: 25%
  corte  3762.00: 56%


**Resposta:** o AID gera 5 faixas, com cortes em 58,30, 96,05, 387,20 e 3.762,00, mas elas são mais difíceis de defender que as de `tenure`.

- **O churn não é monotônico:** 41,5%, depois 65,1%, depois 34,7%.
- **As duas primeiras faixas misturam `tenure` e mensalidade:** nelas, o `tenure` médio é de 1,2 e 1,8 meses, e o que muda é a mensalidade média (R$ 30,60 contra R$ 59,60). Ou seja, elas trazem de volta a `MonthlyCharges`, que o grupo decidiu deixar de fora.
- **Os cortes são instáveis:** só o de 96,05 aparece com frequência (82% das reamostragens). Os outros ficam entre 25% e 56%.

### 5. `TotalCharges` acrescenta algo depois de discretizar `tenure`?

In [10]:
ct = pd.crosstab(faixas(hist["tenure_num"], CORTES_TENURE), faixas(hist["TotalCharges_imp"], cortes_tc))
print("Cramér's V entre as faixas:", round(np.sqrt(chi2_contingency(ct)[0] / (len(hist) * (min(ct.shape) - 1))), 3))

so_tenure = avaliar([("tenure_num", "tenure", por_aid("tenure_num", FAIXAS_TENURE))])
so_tc = avaliar([("TotalCharges_imp", "tc", por_aid("TotalCharges_imp"))])
as_duas = avaliar([("tenure_num", "tenure", por_aid("tenure_num", FAIXAS_TENURE)), ("TotalCharges_imp", "tc", por_aid("TotalCharges_imp"))])
print(f"Só tenure:         AUC {so_tenure.mean():.4f}")
print(f"Só TotalCharges:   AUC {so_tc.mean():.4f} | diferença {np.mean(so_tc - so_tenure):+.4f} ± {np.std(so_tc - so_tenure):.4f}")
print(f"As duas:           AUC {as_duas.mean():.4f} | diferença {np.mean(as_duas - so_tenure):+.4f} ± {np.std(as_duas - so_tenure):.4f}")

Cramér's V entre as faixas: 0.563


Só tenure:         AUC 0.8448
Só TotalCharges:   AUC 0.8416 | diferença -0.0033 ± 0.0043
As duas:           AUC 0.8454 | diferença +0.0006 ± 0.0019


**Resposta:** não. Com as duas, a AUC muda só +0,0006 ± 0,0019, dentro do ruído. Sozinha, `TotalCharges` também não supera `tenure` (−0,0033 ± 0,0043). **`TotalCharges` é candidata a sair**, como o notebook 05 já indicava: não acrescenta previsão e tem cortes instáveis.

### 6. O AID perde desempenho em relação aos decis?

In [11]:
dec = avaliar([("tenure_num", "tenure", por_decis("tenure_num"))])
print(f"Decis (10 faixas):  AUC {dec.mean():.4f}")
print(f"AID ({FAIXAS_TENURE} faixas):      AUC {so_tenure.mean():.4f} | diferença {np.mean(so_tenure - dec):+.4f} ± {np.std(so_tenure - dec):.4f}")

Decis (10 faixas):  AUC 0.8453
AID (6 faixas):      AUC 0.8448 | diferença -0.0004 ± 0.0018


**Resposta:** não de forma relevante. A diferença é de −0,0004 ± 0,0018, com 6 faixas em vez de 10. O AID ainda dá um IV maior (0,80 contra 0,77), porque os cortes são escolhidos pelo alvo. Com o mesmo desempenho e menos dummies, ficamos com o AID.

### 7. Quais são as dummies finais?

Regras:

- **Referência:** cada variável perde uma categoria, a mais frequente no `history`.
- **Serviços adicionais:** ficam só com `Yes`. `No internet service` já está em `InternetService=No`.
- **Telefone:** `PhoneService` sai, porque `MultipleLines=No phone service` já contém essa informação.
- **Categóricas:** nenhuma precisa de agrupamento. A maior tem 4 categorias, e a menor categoria tem 563 clientes.

In [12]:
def dummies_finais(d, cortes=CORTES_TENURE, ref=None):
    ref = faixa_referencia(hist["tenure_num"], cortes) if ref is None else ref
    return pd.concat([dummies_categoricas(d), dummies_faixas(d["tenure_num"], cortes, "tenure", ref)], axis=1)


X = dummies_finais(hist)
X_inf = dummies_finais(inf)
print(pd.DataFrame({
    "clientes": X.sum().astype(int),
    "taxa_churn": [round(y[X[c] == 1].mean(), 3) for c in X.columns],
    "pct_inference": X_inf.mean().round(3),
}).to_string())
print("\nReferências:")
for col, vals in DUMMIES_CATEG.items():
    print(f"  {col}: {sorted(str(v) for v in hist[col].unique() if v not in vals)}")
print("  tenure:", faixas(hist["tenure_num"], CORTES_TENURE).value_counts().idxmax())
matriz = np.c_[np.ones(len(X)), X.values]
print(f"\nDummies: {X.shape[1]} | posto da matriz com intercepto: {np.linalg.matrix_rank(matriz)} de {matriz.shape[1]}")
print("Mesmas colunas no inference:", list(X.columns) == list(X_inf.columns), "| vazios no inference:", int(X_inf.isna().sum().sum()))

                                         clientes  taxa_churn  pct_inference
gender=Female                                2798       0.272          0.489
SeniorCitizen=1                               928       0.423          0.152
Partner=Yes                                  2726       0.201          0.479
Dependents=Yes                               1685       0.153          0.302
MultipleLines=Yes                            2367       0.290          0.429
MultipleLines=No phone service                563       0.254          0.084
InternetService=DSL                          1924       0.194          0.353
InternetService=No                           1233       0.072          0.208
OnlineSecurity=Yes                           1597       0.150          0.299
OnlineBackup=Yes                             1943       0.223          0.344
DeviceProtection=Yes                         1931       0.234          0.348
TechSupport=Yes                              1610       0.153          0.308

**Resposta:** são **25 dummies**: 20 das variáveis categóricas e 5 de `tenure`. A matriz não tem colinearidade perfeita (posto 26 de 26, contando o intercepto), e o `inference` gera as mesmas colunas, sem vazios.

- **Referências:** `Male`, não idoso, sem parceiro, sem dependentes, uma linha, fibra, sem o serviço adicional (ou sem internet), contrato mensal, fatura digital, cheque eletrônico e `tenure` de 17 a 43 meses.
- **Fora das dummies:** `MonthlyIncome` e `MonthlyCharges` (decisão do grupo), `PhoneService` (repetida) e `TotalCharges` (pergunta 5).
- **Sem poder individual:** `gender` e `MultipleLines` têm IV baixo no notebook 04, mas continuam aqui. A seleção de variáveis fica para a etapa do modelo.

## Limites desta análise

- **O AID usa o alvo:** os cortes precisam ser recalculados dentro de cada fold ao avaliar o modelo, como fizemos aqui, para não inflar o desempenho. Os cortes finais usam todo o `history`.
- **Parâmetros escolhidos por nós:** o mínimo de 5% por faixa e o p < 0,05 são escolhas deste estudo, e outros valores geram outros cortes.
- **Clientes com `tenure = 0`:** os 10 casos têm churn de 0%, mas caem na faixa de até 5 meses, de 54,1%. O mínimo de 5% por faixa não permite separá-los. Eles são poucos no `history` e só 1 no `inference`, mas vale lembrar disso ao pontuar clientes recém-chegados.
- **Definição de `tenure`:** se ele para de contar no cancelamento, parte do seu poder pode não estar disponível antes de o cliente sair. Falta a TS confirmar.
- **O teste usa regressão logística:** outro modelo pode reagir de forma diferente a estas faixas.